In [ ]:
import os

# A mappa neve
folder = "datafiles"

# Fájlok és mappák listázása
files = os.listdir(folder)

print(files)


In [ ]:
import pandas as pd

for file in files:
    name = file.split('.')[0]
    print(f"\n{name}:")

    globals()[f"{name}"] = pd.read_csv(f"datafiles/{file}")
    display(globals()[f"{name}"].sample(5))

In [ ]:
import pandas as pd

# Csak a befejezett versenyek (pozíció nem null)
results_finished = results[results['positionText'] != 'R'].copy()
results_finished['position'] = pd.to_numeric(results_finished['position'], errors='coerce')

# Qualifying és results összekapcsolása
df = qualifying.merge(results_finished, on=['raceId', 'driverId', 'constructorId'], how='inner')
print(f"Merge után (qualifying + results): {len(df)} sor")

# Verseny infók hozzáadása
df = df.merge(races[['raceId', 'year', 'name', 'circuitId']], on='raceId', how='left')
print(f"Verseny infók után: {len(df)} sor")

# Pálya infók hozzáadása
df = df.merge(circuits[['circuitId', 'name', 'country']], on='circuitId', how='left', suffixes=('_race', '_circuit'))
print(f"Pálya infók után: {len(df)} sor")

# Versenyző infók
df = df.merge(drivers[['driverId', 'forename', 'surname', 'nationality']], on='driverId', how='left')
df['driver_name'] = df['forename'] + ' ' + df['surname']
df['driver_nationality'] = df['nationality']
print(f"Versenyző infók után: {len(df)} sor")

# Versenyző tapasztalata (eddigi versenyek száma AZ AKTUÁLIS VERSENY ELŐTT)
# Először hozzáadjuk a dátumot a results táblához
results_with_date = results.merge(races[['raceId', 'date']], on='raceId', how='left')

# Minden versenyző-verseny kombinációhoz számoljuk az előző versenyek számát
driver_exp_list = []
for idx, row in df.iterrows():
    current_driver = row['driverId']
    current_race_id = row['raceId']
    current_date = races[races['raceId'] == current_race_id]['date'].iloc[0]
    
    # Összes korábbi verseny száma
    previous_races = results_with_date[
        (results_with_date['driverId'] == current_driver) & 
        (results_with_date['date'] < current_date)
    ]
    
    driver_exp_list.append({
        'raceId': current_race_id,
        'driverId': current_driver,
        'driver_all_races': len(previous_races)
    })

driver_exp_df = pd.DataFrame(driver_exp_list)
df = df.merge(driver_exp_df, on=['raceId', 'driverId'], how='left')
print(f"Driver experience után: {len(df)} sor")

# Tapasztalat kategória
df['experience_category'] = pd.cut(df['driver_all_races'], 
                                     bins=[-1, 10, 50, 200, 1000], 
                                     labels=['Rookie', 'Junior', 'Experienced', 'Veteran'])

# Konstruktőr infók
df = df.merge(constructors[['constructorId', 'name', 'nationality']], on='constructorId', how='left', suffixes=('', '_constructor'))
df['constructor_name'] = df['name']
df['constructor_nationality'] = df['nationality_constructor']
print(f"Konstruktőr infók után: {len(df)} sor")

# Pozíció változás számítás
df['position_change'] = df['position_x'] - df['position_y']  # position_x=quali, position_y=race

# Leggyorsabb kör relatív ideje (verseny leggyorsabbjához képest)
# Először meg kell tisztítani a fastestLapTime mezőt
df['fastestLapTime_clean'] = df['fastestLapTime']

def laptime_to_seconds(laptime_str):
    """Konvertál '1:23.456' formátumot másodpercekre"""
    if pd.isna(laptime_str) or laptime_str == '\\N':
        return None
    try:
        parts = laptime_str.split(':')
        minutes = int(parts[0])
        seconds = float(parts[1])
        return minutes * 60 + seconds
    except:
        return None

df['fastestLapTime_seconds'] = df['fastestLapTime_clean'].apply(laptime_to_seconds)

# Átlagos köridő a versenyen (lap_times alapján)
lap_times_avg = lap_times.groupby(['raceId', 'driverId'])['milliseconds'].mean().reset_index()
lap_times_avg['avg_lap_time'] = lap_times_avg['milliseconds'] / 1000  # másodpercre
lap_times_avg = lap_times_avg[['raceId', 'driverId', 'avg_lap_time']]

df = df.merge(lap_times_avg, on=['raceId', 'driverId'], how='left')
print(f"Köridők után: {len(df)} sor")

# Versenyenkénti leggyorsabb kör
race_fastest = df.groupby('raceId')['fastestLapTime_seconds'].min().reset_index()
race_fastest.columns = ['raceId', 'race_fastest_lap']

df = df.merge(race_fastest, on='raceId', how='left')
print(f"Leggyorsabb körök után infók után: {len(df)} sor")
df['fastest_lap_diff'] = df['fastestLapTime_seconds'] - df['race_fastest_lap']

# Évtized kategória
df['decade'] = (df['year'] // 10) * 10

# Korszakok (era) manuális kategorizálás
def get_era(year):
    if year <= 1960:
        return 'Early F1 (1950-1960)'
    elif year <= 1983:
        return 'Pre-Turbo (1961-1983)'
    elif year <= 1988:
        return 'Turbo Era (1984-1988)'
    elif year <= 2013:
        return 'V8/V10 Era (1989-2013)'
    elif year <= 2021:
        return 'Hybrid Era (2014-2021)'
    else:
        return 'Ground Effect Era (2022-)'

df['era'] = df['year'].apply(get_era)

# Grid pozíció kategória
df['grid_category'] = pd.cut(df['grid'], bins=[0, 3, 10, 30], labels=['Top3', 'Midfield', 'Back'])

# Kontinens (egyszerűsített)
continent_map = {
    'UK': 'Europe', 'Italy': 'Europe', 'Germany': 'Europe', 'France': 'Europe', 
    'Spain': 'Europe', 'Monaco': 'Europe', 'Belgium': 'Europe', 'Austria': 'Europe',
    'Netherlands': 'Europe', 'Portugal': 'Europe', 'Hungary': 'Europe', 'Russia': 'Europe',
    'USA': 'Americas', 'Brazil': 'Americas', 'Canada': 'Americas', 'Mexico': 'Americas',
    'Argentina': 'Americas', 'United States': 'Americas',
    'Japan': 'Asia', 'China': 'Asia', 'Malaysia': 'Asia', 'Singapore': 'Asia',
    'Korea': 'Asia', 'Bahrain': 'Asia', 'UAE': 'Asia', 'Saudi Arabia': 'Asia',
    'Turkey': 'Asia', 'India': 'Asia', 'Azerbaijan': 'Asia', 'Qatar': 'Asia',
    'Australia': 'Oceania',
    'South Africa': 'Africa'
}
df['continent'] = df['country'].map(continent_map).fillna('Other')

# Versenyző átlagos pitstop ideje a versenyen
pitstop_avg = pit_stops.groupby(['raceId', 'driverId'])['milliseconds'].mean().reset_index()
pitstop_avg['avg_pitstop_time'] = pitstop_avg['milliseconds'] / 1000  # másodpercre
pitstop_avg = pitstop_avg[['raceId', 'driverId', 'avg_pitstop_time']]

df = df.merge(pitstop_avg, on=['raceId', 'driverId'], how='left')
print(f"Átlag pitstop infók után: {len(df)} sor")

# Konstruktőr és versenyző pontjai a szezonban az aktuális versenyig
constructor_season_points = []
driver_season_points = []

for idx, row in df.iterrows():
    current_race_id = row['raceId']
    current_year = row['year']
    current_date = races[races['raceId'] == current_race_id]['date'].iloc[0]
    current_constructor = row['constructorId']
    current_driver = row['driverId']
    
    # Konstruktőr pontjai a szezonban eddig
    prev_constructor_races = results_with_date[
        (results_with_date['constructorId'] == current_constructor) &
        (results_with_date['date'] < current_date) &
        (results_with_date['date'].str.startswith(str(current_year)))
    ]
    constructor_points_so_far = prev_constructor_races['points'].sum()
    
    # Versenyző pontjai a szezonban eddig
    prev_driver_races = results_with_date[
        (results_with_date['driverId'] == current_driver) &
        (results_with_date['date'] < current_date) &
        (results_with_date['date'].str.startswith(str(current_year)))
    ]
    driver_points_so_far = prev_driver_races['points'].sum()
    
    constructor_season_points.append({
        'raceId': current_race_id,
        'constructorId': current_constructor,
        'constructor_points_season': constructor_points_so_far
    })
    
    driver_season_points.append({
        'raceId': current_race_id,
        'driverId': current_driver,
        'driver_points_season': driver_points_so_far
    })

constructor_points_df = pd.DataFrame(constructor_season_points)
constructor_points_df = constructor_points_df.drop_duplicates(subset=['raceId', 'constructorId'])

driver_points_df = pd.DataFrame(driver_season_points)
driver_points_df = driver_points_df.drop_duplicates(subset=['raceId', 'driverId'])

df = df.merge(constructor_points_df, on=['raceId', 'constructorId'], how='left')
print(f"Konstr. pontok után: {len(df)} sor")
df = df.merge(driver_points_df, on=['raceId', 'driverId'], how='left')
print(f"Versenyző pontok után: {len(df)} sor")

# Versenyző átlagos grid és race pozíciója eddig
driver_avg_positions = []

for idx, row in df.iterrows():
    current_driver = row['driverId']
    current_race_id = row['raceId']
    current_date = races[races['raceId'] == current_race_id]['date'].iloc[0]
    
    # Korábbi versenyek
    prev_races = results_with_date[
        (results_with_date['driverId'] == current_driver) &
        (results_with_date['date'] < current_date)
    ].copy()
    
    # Grid és position tisztítása: csak numerikus értékek
    prev_races['grid'] = pd.to_numeric(prev_races['grid'], errors='coerce')
    prev_races['position'] = pd.to_numeric(prev_races['position'], errors='coerce')
    
    avg_grid = prev_races['grid'].mean() if len(prev_races) > 0 else None
    avg_position = prev_races['position'].mean() if len(prev_races) > 0 else None
    
    driver_avg_positions.append({
        'raceId': current_race_id,
        'driverId': current_driver,
        'avg_grid_position': avg_grid,
        'avg_race_position': avg_position
    })

driver_avg_df = pd.DataFrame(driver_avg_positions)
driver_avg_df = driver_avg_df.drop_duplicates(subset=['raceId', 'driverId'])

df = df.merge(driver_avg_df, on=['raceId', 'driverId'], how='left')
print(f"Versenyző avg után: {len(df)} sor")

# Pályák DNF aránya ÉS átlagos pozícióváltozás (historikus, az aktuális verseny ELŐTTI adatok alapján)
results_with_race = results.merge(races[['raceId', 'circuitId', 'date']], on='raceId')
results_with_race['is_dnf'] = (results_with_race['positionText'] == 'R').astype(int)

# Minden versenyhez kiszámoljuk az addig történt DNF arányt és pozícióváltozást azon a pályán
circuit_history = []

for circuit_id in results_with_race['circuitId'].unique():
    circuit_races = results_with_race[results_with_race['circuitId'] == circuit_id].sort_values('date')
    
    for i, race_id in enumerate(circuit_races['raceId'].unique()):
        # Csak az EZT MEGELŐZŐ versenyek számítanak
        previous_races = circuit_races[circuit_races['raceId'].isin(
            circuit_races['raceId'].unique()[:i]
        )].copy()
        
        # DNF arány
        if len(previous_races) > 0:
            dnf_rate = previous_races['is_dnf'].mean()
        else:
            dnf_rate = None
        
        # Átlagos pozícióváltozás
        if len(previous_races) > 0:
            previous_races['grid'] = pd.to_numeric(previous_races['grid'], errors='coerce')
            previous_races['position'] = pd.to_numeric(previous_races['position'], errors='coerce')
            
            prev_with_positions = previous_races.dropna(subset=['grid', 'position'])
            
            if len(prev_with_positions) > 0:
                prev_with_positions['pos_change'] = prev_with_positions['grid'] - prev_with_positions['position']
                avg_pos_change = prev_with_positions['pos_change'].mean()
            else:
                avg_pos_change = None
        else:
            avg_pos_change = None
        
        circuit_history.append({
            'raceId': race_id,
            'circuit_dnf_rate': dnf_rate,
            'circuit_avg_position_change': avg_pos_change
        })

circuit_history_df = pd.DataFrame(circuit_history)
df = df.merge(circuit_history_df, on='raceId', how='left')
print(f"Pálya történelem infók után: {len(df)} sor")

# Végleges adattábla
final_df = df[[
    'raceId', 'name_race', 'year', 'decade', 'era', 'name_circuit', 'country', 'continent',
    'circuit_dnf_rate', 'circuit_avg_position_change',
    'driverId', 'driver_name', 'driver_nationality', 'driver_all_races', 'experience_category',
    'driver_points_season', 'avg_grid_position', 'avg_race_position',
    'constructorId', 'constructor_name', 'constructor_nationality', 'constructor_points_season',
    'position_x', 'position_y', 'grid', 'grid_category',
    'position_change', 'points', 'fastestLap', 'rank',
    'fastest_lap_diff', 'avg_lap_time', 'laps', 'avg_pitstop_time'
]].rename(columns={
    'name_race': 'race_name',
    'name_circuit': 'circuit_name',
    'position_x': 'quali_position',
    'position_y': 'race_position'
})

# Mentés
final_df.to_csv('f1_quali_vs_race.csv', index=False)

print(f"Kész! {len(final_df)} sor adattal.")
print("\nElső 5 sor:")
display(final_df.sample(5))
print("\nOszlopok:")
display(final_df.info())

In [ ]:
import pandas as pd

f1 = pd.read_csv('f1_quali_vs_race.csv')
f1.groupby("continent")["country"].unique()